# Personal Finance Tracker

A Python application to manage personal income and expenses.

### Main Features
- Record income and expenses.
- Calculate monthly income, expenses and balance.
- Categorise spending.
- Save transactions in a CSV file.
- Convert currencies using an external API.
- Display a monthly spending summary.

In [ ]:
from datetime import date
import csv
import json
import matplotlib.pyplot as plt
from urllib.request import Request, urlopen
from urllib.error import URLError, HTTPError

## Step 1: Creating an Expense Class

A class helps to keep the information and functions in an organised manner. In the Personal Finance Tracker an `Expense` class to store details such as the category, amount and description of an expense.

In [ ]:
class Expense:
    def __init__(self, category, amount, currency, converted_amount, rate, expense_date, description):
        self.category = category
        self.amount = amount
        self.currency = currency
        self.converted_amount = converted_amount
        self.rate = rate
        self.date = expense_date
        self.description = description

    # Storing the expense details in a dictionary
    def to_dictionary(self):
        return {
            "category": self.category,
            "amount": self.amount,
            "currency": self.currency,
            "converted_amount": self.converted_amount,
            "rate": self.rate,
            "date": self.date,
            "description": self.description
        }


## Step 2: Setting Up and Saving Monthly Finances

In this step, the user chooses a main currency and enters their monthly income. The income can come from different sources like Salary, Freelance or Teaching, etc.

The `save_settings()` function below saves the currency and income sources in a JSON file. The `load_settings()` function loads these settings when the program starts again.

The program also checks if the saved settings are valid or not. If the settings cannot be loaded then the user is asked to enter them again. If an older settings file has only one income amount, it is saved under "Other".

The total monthly income is calculated by adding all the income sources together.

In [ ]:
INCOME_SOURCE_OPTIONS = ["Salary", "Freelance", "Teaching", "Business", "Benefits", "Other"]

# Saving the currency and income sources
def save_settings():
    settings = {"currency": currency, "income_sources": income_sources}
    with open("settings.json", "w", encoding="utf-8") as file:
        json.dump(settings, file, indent=4)

# Load the saved settings
def load_settings():
    global currency, income, income_sources

    try:
        with open("settings.json", "r", encoding="utf-8") as file:
            settings = json.load(file)

        currency = settings["currency"]

        if currency not in ["EUR", "USD", "GBP"]:
            raise ValueError("Invalid currency")

        if "income_sources" in settings:
            saved_sources = settings["income_sources"]

            if not isinstance(saved_sources, dict):
                raise ValueError("Invalid income sources")

            income_sources = {}

            for source, amount in saved_sources.items():
                if not isinstance(source, str) or not source:
                    raise ValueError("Invalid income source")

                amount = float(amount)

                if amount < 0:
                    raise ValueError("Income cannot be negative")

                income_sources[source] = amount
                
        # Keeping the income from an older settings file
        else:
            old_income = float(settings["income"])

            if old_income < 0:
                raise ValueError("Income cannot be negative")

            income_sources = {"Other": old_income}

        income = sum(income_sources.values())
        print("Saved settings loaded successfully.")
        
    # Asking the user for the settings if they cannot be loaded
    except (FileNotFoundError, json.JSONDecodeError, KeyError, ValueError, TypeError):
        print("Set up your monthly finances.")
        print("Choose your main currency:")
        print("1. EUR")
        print("2. USD")
        print("3. GBP")

        currencies = ["EUR", "USD", "GBP"]

        while True:
            choice = input("Enter 1, 2 or 3: ")

            if choice in ["1", "2", "3"]:
                currency = currencies[int(choice) - 1]
                break

            print("Please choose 1, 2 or 3.")
            
        # Enter the months income -- by the user
        while True:
            try:
                income = float(input("Enter your monthly income in {}: ".format(currency)))

                if income >= 0:
                    break

                print("Income cannot be negative.")

            except ValueError:
                print("Please enter a valid amount.")

        income_sources = {"Other": income}
        save_settings()
        print("Settings saved successfully.")

load_settings()

expenses = []

print("Your monthly finances are ready.")
print("Monthly income: {} {:.2f}".format(currency, income))


## Step 3: Currency Conversion

The program here converts an expense from one currency to another using exchange rates from an external API.

The `convert_currency()` function is taking the amount, the original currency and the main currency as inputs.

If both currencies are same then the amount does not need to be converted. Otherwise the program gets the exchange rates from the API and calculates the converted amount.

If the exchange rates cannot be retrieved, the program will display an error message and will not continue with the conversion.

In [ ]:
def convert_currency(amount, from_currency, to_currency):
    # No conversion is needed if the currencies are the same
    if from_currency == to_currency:
        return amount, 1.0 # 1.0 means the exchange rate is 1, so the amount stays the same

    url = "https://open.er-api.com/v6/latest/USD"
    request = Request(url,headers={"User-Agent": "PersonalFinanceTracker/1.0"})

    try:
        # Getting the exchange rates
        with urlopen(request, timeout=20) as response:
            data = json.loads(response.read().decode("utf-8"))

        if data["result"] != "success":
            print("Could not get exchange rates.")
            return None, None

        rates = data["rates"]
        from_rate = float(rates[from_currency])
        to_rate = float(rates[to_currency])

        # Converting the amount into the main currency
        rate = to_rate / from_rate
        converted_amount = amount * rate

        return converted_amount, rate

    except (HTTPError, URLError, TimeoutError, OSError, KeyError, ValueError) as error:
        print("Could not get exchange rate: {}".format(error))
        return None, None

In [ ]:
amount = 50

# Converting USD into EUR
converted_amount, rate = convert_currency(amount, "USD", "EUR")

if converted_amount is not None:
    print("Original amount: USD {:.2f}".format(amount))
    print("Exchange rate: 1 USD = {:.4f} EUR".format(rate))
    print("Converted amount: EUR {:.2f}".format(converted_amount))

## Step 4: Saving and Loading Expenses

This code below save the expenses in a CSV file called `expenses.csv`. It means the expenses can still be used when the program is opened again.

The `save_expenses()` function writes the expense details into the CSV file. The `load_expenses()` function reads the saved expenses and creates `Expense` objects from them.

The program below also checks the saved data. If a row is invalid then it skips that row instead of stopping the whole program. If the CSV file does not exist then the program will create a new one.

In [ ]:
def save_expenses():
    # Saving all expenses in the CSV file
    with open("expenses.csv", "w", newline="", encoding="utf-8") as file:
        columns = ["category", "amount", "currency", "converted_amount", "rate", "date", "description"]
        writer = csv.DictWriter(file, fieldnames=columns)
        writer.writeheader()

        for expense in expenses:
            writer.writerow(expense.to_dictionary())

def load_expenses():
    global expenses
    loaded_expenses = []

    try:
        # Reading the saved expenses
        with open("expenses.csv", "r", newline="", encoding="utf-8") as file:
            reader = csv.DictReader(file)

            for item in reader:
                try:
                    category = item["category"]
                    amount = float(item["amount"])
                    expense_currency = item.get("currency") or currency
                    expense_date = item["date"]

                    # Checking the expense details
                    if not category or not expense_date or amount <= 0:
                        raise ValueError

                    if expense_currency not in ["EUR", "USD", "GBP"]:
                        raise ValueError

                    # Use the saved amount and rate
                    converted_amount = float(item.get("converted_amount") or amount)
                    rate = float(item.get("rate") or 1.0)
                    description = item.get("description") or ""

                    if converted_amount < 0 or rate <= 0:
                        raise ValueError

                    # Creating an expense object
                    expense = Expense(category, amount, expense_currency, converted_amount, rate, expense_date, description)
                    loaded_expenses.append(expense)

                except (ValueError, KeyError, TypeError):
                    print("Skipping an invalid expense row.")

        expenses = loaded_expenses
        print("Saved expenses loaded successfully.")
        print("Number of saved expenses: {}".format(len(expenses)))

    except FileNotFoundError:
        expenses = []
        save_expenses()
        print("A new expense file has been created.")

    except (OSError, csv.Error) as error:
        print("Could not load expenses: {}".format(error))

In [ ]:
load_expenses()
print("Number of saved expenses: {}".format(len(expenses)))

## Step 5: Record Expenses

In this step the user can add a new expense by choosing a category and currency and entering an amount. A description can also be added if needed but it is optional.

The program below checks the user's input and converts the amount into the main currency. It then saves the expense in the CSV file. Finally, the program calculates the total spending for the current month and shows the remaining balance.

In [ ]:
def add_expense():
    # Choosing an expense category
    categories = ["Food", "Groceries", "Shopping", "Rent", "Transport", "Bills", "Education", "Healthcare", "Entertainment", "Other"]

    print("\nChoose an expense category:")

    for i, category in enumerate(categories, start=1):
        print("{}. {}".format(i, category))

    while True:
        choice = input("Enter category number: ")

        if choice.isdigit() and 1 <= int(choice) <= len(categories):
            category = categories[int(choice) - 1]
            break

        print("Please choose a valid number.")

    # Choose the currency
    currencies = ["EUR", "USD", "GBP"]

    print("\nChoose expense currency:")

    for i, item in enumerate(currencies, start=1):
        print("{}. {}".format(i, item))

    while True:
        choice = input("Enter currency number: ")

        if choice.isdigit() and 1 <= int(choice) <= len(currencies):
            expense_currency = currencies[int(choice) - 1]
            break

        print("Please choose a valid number.")

    # Enter the expense amount
    while True:
        try:
            amount = float(input("Enter expense amount: "))

            if amount > 0:
                break

            print("Amount must be greater than zero.")

        except ValueError:
            print("Please enter a valid amount.")

    # Converting the amount into the main currency
    converted_amount, rate = convert_currency(amount, expense_currency, currency)

    if converted_amount is None:
        print("Expense was not saved because conversion failed.")
        return

    note = input("Description (optional): ").strip()

    # Creating and saving the expense
    expense = Expense(category, amount, expense_currency, round(converted_amount, 2), rate, date.today().isoformat(), note)

    expenses.append(expense)
    save_expenses()

    # Calculating this month's spending
    this_month = date.today().strftime("%Y-%m")
    spent = 0

    for item in expenses:
        if item.date[:7] == this_month:
            spent = spent + item.converted_amount

    balance = income - spent

    print("\nExpense recorded successfully!")
    print("Date: {}".format(expense.date))
    print("Category: {}".format(expense.category))
    print("Original amount: {} {:.2f}".format(expense.currency, expense.amount))
    print("Converted amount: {} {:.2f}".format(currency, expense.converted_amount))
    print("Monthly spending: {} {:.2f}".format(currency, spent))
    print("Remaining balance: {} {:.2f}".format(currency, balance))

## Step 6: Testing Expense Recording

In this we test the `add_expense()` function to check if an expense can be added and saved correctly.

The function below asks for the expense details, converts the amount into the main currency and saves it in the CSV file and shows the updated monthly spending and remaining balance.

The test is kept separate and commented out so that a new expense is not added every time all the notebook cells are run.

In [ ]:
# Run this cell for testing
#add_expense()

## Step 7: Monthly Spending Summary

The code below calculates the total expenses for the current month and subtracts them from the monthly income to find the remaining balance. It also groups expenses by category like Food, Rent and Shopping, etc and shows how much was spent in each category. If no expenses were recorded this month, the program will displays a message.

In [ ]:
def show_summary():
    this_month = date.today().strftime("%Y-%m")
    spent = 0
    category_totals = {}

    # Calculating this month's expenses
    for expense in expenses:
        if expense.date[:7] == this_month:
            amount = expense.converted_amount
            spent = spent + amount
            category = expense.category

            # Adding expenses from the same category together
            if category in category_totals:
                category_totals[category] = category_totals[category] + amount
            else:
                category_totals[category] = amount

    # Calculating the remaining balance
    balance = income - spent

    print("\nMonthly Summary - {}".format(date.today().strftime("%B %Y")))
    print("--------------------------------")
    print("Monthly income: {} {:.2f}".format(currency, income))
    print("Total expenses: {} {:.2f}".format(currency, spent))
    print("Remaining balance: {} {:.2f}".format(currency, balance))

    # Show spending for each category
    print("\nSpending by category:")

    if len(category_totals) == 0:
        print("No expenses recorded this month.")
    else:
        for category in category_totals:
            print("{}: {} {:.2f}".format(category, currency, category_totals[category]))

## Step 8: Verifying Saved Expenses

Checking if the saved expenses can be loaded from the CSV file. The program then display the monthly summary to check if the expenses and remaining balance are calculated correctly or not.

In [ ]:
load_expenses()
show_summary()

## Step 9: Visualise Spending by Category

The code below creates a bar chart to show the total spending in each category for the current month. It adds together expenses from the same category and displays the results in a chart. The amounts are shown in the main currency. If there are no expenses for the current month, the program displays a message instead.

In [ ]:
this_month = date.today().strftime("%Y-%m")
category_totals = {}

# Adding up expenses for each category
for expense in expenses:
    if expense.date[:7] == this_month:
        category = expense.category
        amount = expense.converted_amount

        if category in category_totals:
            category_totals[category] = category_totals[category] + amount
        else:
            category_totals[category] = amount

# Showing the chart if there are expenses
if len(category_totals) == 0:
    print("No expenses recorded this month.")
else:
    categories = list(category_totals.keys())
    amounts = list(category_totals.values())

    plt.figure(figsize=(9, 5))
    plt.bar(categories, amounts)

    plt.title("Monthly Spending by Category")
    plt.xlabel("Expense Category")
    plt.ylabel("Amount ({})".format(currency))

    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## Step 10: Creating the Graphical User Interface (GUI)

Below Tkinter is used to create the graphical user interface (GUI) for the Personal Finance Tracker. The GUI allows the user to enter expenses, choose categories and currencies, update income and view the monthly summary and spending chart.

In [ ]:
import tkinter as tk
from tkinter import ttk

print("Tkinter is working")

## Step 11: Creating the GUI and Expense Form

The code below has created the main window of the Personal Finance Tracker using Tkinter.

The window has a category dropdown, an amount field, a currency dropdown and an optional description field also colours and styles are added to make the application look better.

The income section, summary and buttons are added in the next steps.

In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox

root = tk.Tk()
root.title("Personal Finance Tracker")
root.geometry("700x850")
root.configure(bg="#EDF4FA")

# Setting the colours and styles
style = ttk.Style()
style.theme_use("clam")

style.configure("TFrame", background="#EDF4FA")
style.configure("TLabel", background="#EDF4FA", foreground="#243B53", font=("Arial", 11))
style.configure("Title.TLabel", background="#17365D", foreground="white", font=("Arial", 20, "bold"), padding=12)
style.configure("Section.TLabel", background="#EDF4FA", foreground="#17365D", font=("Arial", 14, "bold"))

style.configure("TButton", font=("Arial", 10, "bold"), padding=7, background="#D6E4F0", foreground="#17365D")
style.configure("Primary.TButton", background="#2868A6", foreground="white", padding=9)
style.configure("Chart.TButton", background="#138577", foreground="white", padding=9)
style.configure("Save.TButton", background="#D6E4F0", foreground="#17365D", padding=8)
style.configure("Danger.TButton", background="#FBE5E5", foreground="#A52A36", padding=7)

style.configure("TEntry", fieldbackground="white", foreground="#243B53", padding=5)
style.configure("TCombobox", fieldbackground="white", background="#D6E4F0", foreground="#243B53", padding=5)

# Changing the button colours when clicked or hovered over
style.map("TCombobox", fieldbackground=[("readonly", "white")], selectbackground=[("readonly", "white")], selectforeground=[("readonly", "#243B53")])
style.map("Primary.TButton", background=[("active", "#1D5085")])
style.map("Chart.TButton", background=[("active", "#0D6C60")])
style.map("Danger.TButton", background=[("active", "#F4CACA")])

# Creating the main area
main_frame = ttk.Frame(root, padding=12)
main_frame.pack(fill="both", expand=True)
main_frame.columnconfigure(1, weight=1)

# Adding the heading
heading = ttk.Label(main_frame, text="Personal Finance Tracker", style="Title.TLabel")
heading.grid(row=0, column=0, columnspan=2, sticky="ew", pady=10)

# Choosing an expense category
ttk.Label(main_frame, text="Expense Category:").grid(row=1, column=0, sticky="w", pady=4)

category_box = ttk.Combobox(main_frame, values=["Food", "Groceries", "Shopping", "Rent", "Transport", "Bills", "Education", "Healthcare", "Entertainment", "Other"], state="readonly")
category_box.grid(row=1, column=1, sticky="ew", pady=4)
category_box.set("Food")

# Entering the expense amount
ttk.Label(main_frame, text="Expense Amount:").grid(row=2, column=0, sticky="w", pady=4)

amount_entry = ttk.Entry(main_frame)
amount_entry.grid(row=2, column=1, sticky="ew", pady=4)

# Choosing the currency
ttk.Label(main_frame, text="Currency:").grid(row=3, column=0, sticky="w", pady=4)

currency_box = ttk.Combobox(main_frame, values=["EUR", "USD", "GBP"], state="readonly")
currency_box.grid(row=3, column=1, sticky="ew", pady=4)
currency_box.set(currency)

# Entering an optional description
ttk.Label(main_frame, text="Description (optional):").grid(row=4, column=0, sticky="w", pady=4)

description_entry = ttk.Entry(main_frame)
description_entry.grid(row=4, column=1, sticky="ew", pady=4)

## Step 12: Connecting the GUI to the Tracker

Connecting the form and buttons to the functions that were created earlier in above code.

The user can add expenses, save different income sources and view the monthly summary. The program also checks the entered amounts and shows an error message if something is wrong.

After an expense is added, the program saves it, updates the summary and refreshes the saved-expenses list.

In [ ]:
# Income source dropdown
ttk.Label(main_frame, text="Income Source:").grid(row=6, column=0, sticky="w", pady=4)

income_options = INCOME_SOURCE_OPTIONS.copy()

for source in income_sources:
    if source not in income_options:
        income_options.append(source)

income_source_box = ttk.Combobox(main_frame, values=income_options, state="readonly")
income_source_box.grid(row=6, column=1, sticky="ew", pady=4)
income_source_box.set("Other")

# Monthly income amount
ttk.Label(main_frame, text="Monthly Amount ({}):".format(currency)).grid(row=7, column=0, sticky="w", pady=4)

income_entry = ttk.Entry(main_frame)
income_entry.grid(row=7, column=1, sticky="ew", pady=4)

# Buttons for adding expenses and viewing the chart
action_frame = ttk.Frame(main_frame)
action_frame.grid(row=5, column=0, columnspan=2, pady=4)

# Monthly summary and status message
ttk.Label(main_frame, text="Monthly Summary", style="Section.TLabel").grid(row=9, column=0, columnspan=2, pady=(6, 3))

summary_label = ttk.Label(main_frame, text="", justify="left", wraplength=600)
summary_label.grid(row=10, column=0, columnspan=2, pady=3)

status_label = ttk.Label(main_frame, text="To add an expense, complete the form above and click Add Expense.", wraplength=600, justify="center")
status_label.grid(row=11, column=0, columnspan=2, pady=4)

# Showing the saved amount for the selected source
def show_income_source_amount(event=None):
    source = income_source_box.get()
    income_entry.delete(0, tk.END)

    if source in income_sources:
        income_entry.insert(0, str(income_sources[source]))

income_source_box.bind("<<ComboboxSelected>>", show_income_source_amount)
show_income_source_amount()

# Update the monthly summary
def update_summary():
    this_month = date.today().strftime("%Y-%m")
    spent = 0

    for item in expenses:
        if item.date[:7] == this_month:
            spent = spent + item.converted_amount

    income_details = []

    for source in income_sources:
        if income_sources[source] > 0:
            income_details.append("{} {} {:.2f}".format(source, currency, income_sources[source]))

    if len(income_details) > 0:
        source_text = ", ".join(income_details)
    else:
        source_text = "No income sources recorded"

    balance = income - spent

    summary_label.config(text="Monthly income: {} {:.2f}\nIncome sources: {}\nTotal expenses: {} {:.2f}\nRemaining balance: {} {:.2f}".format(currency, income, source_text, currency, spent, currency, balance))

# Saving the selected income source
def update_income_from_form():
    global income

    source = income_source_box.get()

    try:
        new_income = float(income_entry.get())

        if new_income < 0:
            messagebox.showerror("Invalid Income", "Income cannot be negative.")
            return

    except ValueError:
        messagebox.showerror("Invalid Income", "Please enter a valid amount.")
        return

    old_amount = income_sources.get(source)
    income_sources[source] = new_income
    income = sum(income_sources.values())

    try:
        save_settings()

    except OSError as error:
        if old_amount is None:
            income_sources.pop(source)
        else:
            income_sources[source] = old_amount

        income = sum(income_sources.values())
        update_summary()
        messagebox.showerror("Saving Error", "Could not save income: {}".format(error))
        return

    update_summary()
    status_label.config(text="{} income saved. Total monthly income: {} {:.2f}".format(source, currency, income))

# Adding and saving a new expense
def add_expense_from_form():
    category = category_box.get()
    expense_currency = currency_box.get()
    note = description_entry.get().strip()

    try:
        amount = float(amount_entry.get())

        if amount <= 0:
            messagebox.showerror("Invalid Amount", "Amount must be greater than zero.")
            return

    except ValueError:
        messagebox.showerror("Invalid Amount", "Please enter a valid number.")
        return

    converted_amount, rate = convert_currency(amount, expense_currency, currency)

    if converted_amount is None:
        messagebox.showerror("Currency Error", "The expense was not saved because conversion failed.")
        return

    expense = Expense(category, amount, expense_currency, round(converted_amount, 2), rate, date.today().isoformat(), note)
    expenses.append(expense)

    try:
        save_expenses()

    except (OSError, csv.Error) as error:
        expenses.pop()
        messagebox.showerror("Saving Error", "Could not save the expense: {}".format(error))
        return

    update_summary()
    refresh_expense_list()
    status_label.config(text="Expense added and saved successfully!")

    amount_entry.delete(0, tk.END)
    description_entry.delete(0, tk.END)

# Adding Expense button
add_button = ttk.Button(action_frame, text="+ Add Expense", command=add_expense_from_form, style="Primary.TButton")
add_button.grid(row=0, column=0, padx=6)

# Saving income button
income_button = ttk.Button(main_frame, text="Save / Update Income Source", command=update_income_from_form, style="Save.TButton")
income_button.grid(row=8, column=0, columnspan=2, pady=4)

update_summary()

## Step 13: Displaying the Spending Chart

In this code a button is created which opens a chart showing the current month's spending by category. The program then adds together expenses from the same category and shows the totals as bars. The amount is shown above each bar, and the values use the main currency. If there are no expenses for the current month, a message is shown.

In [ ]:
from matplotlib.figure import Figure
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

def show_spending_chart():
    this_month = date.today().strftime("%Y-%m")
    category_totals = {}

    # Adding up spending in each category
    for expense in expenses:
        if expense.date[:7] == this_month:
            category = expense.category
            amount = expense.converted_amount

            if category in category_totals:
                category_totals[category] = category_totals[category] + amount
            else:
                category_totals[category] = amount

    # Checking if there are expenses to show
    if len(category_totals) == 0:
        messagebox.showinfo("Spending Chart", "No expenses recorded this month.")
        return

    # Creating the chart window
    chart_window = tk.Toplevel(root)
    chart_window.title("Monthly Spending Chart")
    chart_window.geometry("750x500")
    chart_window.configure(bg="#EDF4FA")

    figure = Figure(figsize=(7, 4), dpi=100, facecolor="#EDF4FA")
    chart = figure.add_subplot(111)
    chart.set_facecolor("white")

    categories = list(category_totals.keys())
    amounts = list(category_totals.values())

    # Drawing the bars and showing the amounts
    bars = chart.bar(categories, amounts, color="#138577")
    chart.bar_label(bars, fmt="%.2f", padding=3)
    chart.set_ylim(0, max(amounts) * 1.15)

    chart.set_title("Monthly Spending by Category", color="#17365D")
    chart.set_xlabel("Expense Category")
    chart.set_ylabel("Amount ({})".format(currency))
    chart.tick_params(axis="x", rotation=45)

    # Making the chart easier to read
    chart.grid(axis="y", linestyle="--", alpha=0.3)
    chart.set_axisbelow(True)
    chart.spines["top"].set_visible(False)
    chart.spines["right"].set_visible(False)
    figure.tight_layout()

    # Displaying the chart in the window
    canvas = FigureCanvasTkAgg(figure, master=chart_window)
    canvas.draw()
    canvas.get_tk_widget().pack(fill="both", expand=True)

# Putting the chart button next to Add Expense button
chart_button = ttk.Button(action_frame, text="View Spending Chart", command=show_spending_chart, style="Chart.TButton")
chart_button.grid(row=0, column=1, padx=6)

## Step 14: Managing and Deleting Saved Expenses

In this code, the user can view saved expenses and choose one to delete. The program asks for confirmation before deleting an expense. There is also a button to delete all expenses. After deleting, the program saves the changes, updates the expense list and refreshes the monthly summary.

In [ ]:
# Showing the saved expenses
ttk.Label(main_frame, text="Saved Expenses", style="Section.TLabel").grid(row=12, column=0, columnspan=2, pady=(6, 3))

expense_box = ttk.Combobox(main_frame, state="readonly", width=55)
expense_box.grid(row=13, column=0, columnspan=2, sticky="ew", pady=3)

# Updating the expense dropdown
def refresh_expense_list():
    expense_labels = []

    for item in expenses:
        label = "{} | {} | {} {:.2f} | {}".format(item.date, item.category, item.currency, item.amount, item.description)
        expense_labels.append(label)

    expense_box["values"] = expense_labels

    if len(expense_labels) > 0:
        expense_box.current(0)
    else:
        expense_box.set("")

# Deleting one selected expense
def delete_selected_expense():
    selected_index = expense_box.current()

    if selected_index == -1:
        messagebox.showinfo("Delete Expense", "Please select an expense first.")
        return

    answer = messagebox.askyesno("Delete Expense", "Are you sure you want to delete this expense?")

    if answer:
        deleted_expense = expenses.pop(selected_index)

        try:
            save_expenses()
        except (OSError, csv.Error) as error:
            expenses.insert(selected_index, deleted_expense)
            refresh_expense_list()
            messagebox.showerror("Saving Error", "Could not update the file: {}".format(error))
            return

        refresh_expense_list()
        update_summary()
        status_label.config(text="Expense deleted successfully.")

# Deleting all expenses
def delete_all_expenses():
    if len(expenses) == 0:
        messagebox.showinfo("Delete All Expenses", "There are no expenses to delete.")
        return

    answer = messagebox.askyesno("Delete All Expenses", "Are you sure you want to delete ALL expenses?")

    if answer:
        old_expenses = expenses.copy()
        expenses.clear()

        try:
            save_expenses()
        except (OSError, csv.Error) as error:
            expenses.extend(old_expenses)
            refresh_expense_list()
            messagebox.showerror("Saving Error", "Could not update the file: {}".format(error))
            return

        refresh_expense_list()
        update_summary()
        status_label.config(text="All expenses deleted successfully.")

# Placing the delete buttons together
delete_frame = ttk.Frame(main_frame)
delete_frame.grid(row=14, column=0, columnspan=2, pady=4)

delete_button = ttk.Button(delete_frame, text="Delete Selected", command=delete_selected_expense, style="Danger.TButton")
delete_button.grid(row=0, column=0, padx=5)

delete_all_button = ttk.Button(delete_frame, text="Delete All Expenses", command=delete_all_expenses, style="Danger.TButton")
delete_all_button.grid(row=0, column=1, padx=5)

# Loading the saved expenses
refresh_expense_list()

# Opening the application
root.mainloop()